In [ ]:
### chose to work with git bash here with following code:
### sort -R /mnt/c/pathTo/originalSentenceFile.txt | head -n 50000000 > /mnt/c/pathTo/sampleFile.txt

In [ ]:
###############################
### installing depencencies ###
###############################

# pip install wheel
# pip install -U spacy
# python -m spacy download en_core_web_sm
# pip install unix --upgrade
# pip install panda
# pip install sh

# python -m spacy download pl_core_news_md

In [80]:
###############
### imports ###
###############

import spacy
from spacy.language import Language
from spacy_language_detection import LanguageDetector
from langdetect import detect_langs
from langdetect import detect
import unix
import pandas as pd
import re
import subprocess

In [49]:
def get_lang_detector(nlp, name):
    return LanguageDetector(seed=42)  # We use the seed 42

In [51]:
nlp_pl = spacy.load("pl_core_news_md")
Language.factory("language_detector", func=get_lang_detector)
nlp_pl.add_pipe('language_detector', last=True)

In [75]:
# reading txt files in and creating a list of the sentences
with open('C:/Users/torto/Downloads/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]

### now using spacy to pick out nouns
### here also using LanguageDetector to make sure it's the right language
### LanguageDetector doesn't seem to be reliable though...
for s in sentences:
    sn = nlp_pl(s)
    if sn._.language.get('language') == 'pl':
        for token in sn:
            #print(token.text, token.pos_, token.dep_)
            if token.pos_ == 'NOUN':
                continue
                #print(token.text, token.lemma_)
    elif sn._.language.get('score') >= 0.999:
        print('Sentence', sn, 'is not pl but likely',  sn._.language.get('language'))

Sentence ! "Drugi"? is not pl but likely hr
Sentence ! "Otia"! is not pl but likely sw
Sentence ! "Otia"! is not pl but likely sw
Sentence ! Dalej! is not pl but likely sk
Sentence ! Patterson? is not pl but likely sv
Sentence !" "No to se ugryź". is not pl but likely en
Sentence !" Mniam, mniam. is not pl but likely sw
Sentence !" Mniam, mniam. is not pl but likely sw
Sentence " " Can anybody is not pl but likely cy
Sentence " " I just gotta get out of this prison cell is not pl but likely en
Sentence " " Let us give you a few tricks, some old and then some new tricks is not pl but likely en
Sentence " " We're very versatile is not pl but likely da
Sentence " 'Cause the world keeps spinning round and round " is not pl but likely en
Sentence " 'Cause the world keeps spinning round and round " is not pl but likely en
Sentence " 'Cause tomorrow is a brand-new day " is not pl but likely en
Sentence " 'Cause you would be my Venus of the stars. " is not pl but likely en
Sentence " 'Cause yo

In [89]:
# reading txt files in and creating a list of the sentences
with open('C:/Users/torto/Downloads/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]

### now using spacy to pick out nouns
### this time using the package langdetect
for s in sentences:
    try: print(s, detect_langs(s))
    except Exception as e:
        print(e)
    sn = nlp_pl(s)

    """if sn._.language.get('language') == 'pl':
        for token in sn:
            #print(token.text, token.pos_, token.dep_)
            if token.pos_ == 'NOUN':
                continue
                #print(token.text, token.lemma_)
    elif sn._.language.get('score') >= 0.999:
        print('Sentence', sn, 'is not pl but likely',  sn._.language.get('language'))"""

! "Drugi"? [hr:0.9999943424230597]
! "Odwaga to głupota. [pl:0.8571415736391612, tl:0.14285620795131615]
! "Otia"! [sw:0.9999944521826221]
! "Otia"! [sw:0.9999944521826221]
! "Will?" [sw:0.5714254722667773, sv:0.4285704599551277]
! "Świrus"? [pl:0.999996926890012]
! Chodź, idziemy. [pl:0.9999938883899676]
! Czekajcie! [pl:0.9999962243283356]
! Czwarty Hokage. [pl:0.9999968445358681]
! Dalej! [sk:0.9999942605788475]
! Dzisiejszej nocy, pierwsze zdjęcie obcych w sieci okazało się wirusem. [pl:0.9999985875844687]
! Ichigo? [sw:0.8571403351343397, en:0.1428561471349233]
! Jak się tu dostałeś...? [pl:0.9999986573980112]
! Jesli nie chcesz, zeby cie zobaczyli, jest tunel, ktory biegnie pod cmentarzem. [pl:0.9999975277592708]
! Jestem zmęczony! W więzieniu nie za często korzysta się z prysznicy. [pl:0.9999999161098295]
! London, otwórz! [pl:0.8571415337938524, es:0.14285752412392294]
! Nie dziś, Szatanie! [pl:0.999997917383169]
! Nie radziłbym. [pl:0.9999995110595847]
! Nie radziłbym. [pl:0.9

In [108]:
### reading txt files in and creating a list of the sentences
### this time NO language detection, since it will probably not matter with matching and frequency...
### here also with storage of the data in a pd data frame...

# this is the place where we would also determine the specific meaning
# maybe through a tool, maybe AI

with open('C:/Users/torto/Downloads/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]

df = pd.DataFrame(columns=['text', 'lemma', 'pos'])
### now using spacy to determine the pos tags
### this time without using LanguageDetector
for s in sentences:
    sn = nlp_pl(s)
    for token in sn:
        """if token.pos_ == 'VERB':
            print(token.text, token.lemma_, token.pos_)
        elif token.pos_ == 'NOUN':
            print(token.text, token.lemma_, token.pos_)
        elif token.pos_ == 'ADJ':
            print(token.text, token.lemma_, token.pos_)"""

### maybe we don't even need to distinguish...
### since there are forms that might be two different POS...
### we can actually already include a count ... somehow

        if token.pos_ == 'VERB' or token.pos_ == 'NOUN' or token.pos_ == 'ADJ':
            df.loc[len(df)] = [token.text, token.lemma_, token.pos_]
        else:
            continue
print(df)


ValueError: cannot set a row with mismatched columns

In [109]:
### reading txt files in and creating a list of the sentences
### this time NO language detection, since it will probably not matter with matching and frequency...
### here trying the option to store the data in a dictionary, for accessibility

# this is the place where we would also determine the specific meaning
# maybe through a tool, maybe AI

with open('C:/Users/torto/Downloads/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]
pl_dict = dict()
### now using spacy to determine the pos tags
### this time without using LanguageDetector
for s in sentences:
    sn = nlp_pl(s)

### is dict the way to go??? I'm starting to doubt...
    for token in sn:
        if token.pos_ == 'VERB' or token.pos_ == 'NOUN' or token.pos_ == 'ADJ':
            pl_dict[token.lemma] = [token.text, token.pos_]
        else:
            continue
#print(df)


<class 'dict'>


"for s in sentences:\n    sn = nlp_pl(s)\n    for token in sn:\n\n        if token.pos_ == 'VERB' or token.pos_ == 'NOUN' or token.pos_ == 'ADJ':\n            df.loc[len(df)] = [token.text, token.lemma_, token.pos_]\n        else:\n            continue"

In [ ]:
# function for reading in txt files of the languages from OpenSubtitles
def load_text_to_df(path, encoding='utf-8', strip_hyphens=True):
    with open(path, 'r', encoding=encoding) as file:
        text = file.read()

    if strip_hyphens:
        text = text.replace('-', '')

    # split into sentences (or paragraphs, depending on your data)
    sentences = [s.strip() for s in text.split('\n') if s.strip()]

    # create DataFrame
    df = pd.DataFrame({'sentence': sentences})
    return df

In [34]:
def load_text_to_df_stream(in_path, samplesize, out_path, encoding='utf-8'):
    sentences = []
    with open(in_path, 'r', encoding=encoding) as file:
        for line in file:
            s = line.strip()
            if s:
                sentences.append(s)
        #sentences = random.sample(sentences, samplesize)

        with open(out_path, 'w', encoding='utf-8') as f_out:
            for s in sentences:
                f_out.write(s + '\n')

In [ ]:
load_text_to_df_stream('C:/Users/torto/Downloads/pl/pl.txt', 50000000, 'C:/Users/torto/Downloads/pl/pl_stream1.txt', encoding='utf-8')
load_text_to_df_stream('C:/Users/torto/Downloads/pl/pl.txt', 50000000, 'C:/Users/torto/Downloads/pl/pl_stream1.txt', encoding='utf-8')

In [2]:
subprocess.run(["bash", "-c", "echo hello from bash"])

CompletedProcess(args=['bash', '-c', 'echo hello from bash'], returncode=0)

In [5]:
# new try with sh
def shuf_sample(in_path, out_path, samplesize):
    subprocess.run(["shuf", "-n", str(samplesize), in_path, "-o", out_path], check=True)

subprocess.run(["bash", "-c", "shuf -n 10000 /mnt/c/Users/torto/Downloads/pl/pl.txt > /mnt/c/Users/torto/Downloads/pl/pl_shufSample.txt"])
#subprocess.run(["bash", "-c", "shuf --version"])

CompletedProcess(args=['bash', '-c', 'shuf -n 10000 /mnt/c/Users/torto/Downloads/pl/pl.txt > /mnt/c/Users/torto/Downloads/pl/pl_shufSample.txt'], returncode=0)

In [6]:
shuf_sample('/mnt/c/Users/torto/Downloads/pl/pl.txt', '/mnt/c/Users/torto/Downloads/pl/pl_shufSample.txt',5)

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [30]:
def shuf_sample2(in_path, out_path, samplesize):

    in_path = in_path.replace('\\', '/')
    out_path = out_path.replace('\\', '/')

    if in_path[1:3] == ":/":
        drive = in_path[0].lower()
        in_path_bash = f"/mnt/{drive}/{in_path[3:]}"
        out_path_bash = f"/mnt/{drive}/{out_path[3:]}"
    else:
        in_path_bash = in_path
        out_path_bash = out_path

    cmd = f"shuf -n {samplesize} '{in_path_bash}' -o '{out_path_bash}'"

    result = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)

    if result.returncode != 0:
        print("Error running shuf:")
        print(result.stderr)
    else:
        print(f"✅ Sample of {samplesize} lines saved to {out_path_bash}")

    return result.returncode


In [32]:
shuf_sample2("C:/Users/torto/Downloads/pl/pl.txt", "C:/Users/torto/Downloads/pl/pl_shuf_T.txt", 50000000)

⚠️  Error running shuf:



9